In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [2]:
df = pd.read_csv("../data/Lung Cancer.csv")

# normalize column names
df.columns = df.columns.str.strip().str.lower()

print("Columns:", df.columns.tolist())
print(df.head())


Columns: ['age', 'gender', 'smoking', 'finger_discoloration', 'mental_stress', 'exposure_to_pollution', 'long_term_illness', 'energy_level', 'immune_weakness', 'breathing_issue', 'alcohol_consumption', 'throat_discomfort', 'oxygen_saturation', 'chest_tightness', 'family_history', 'smoking_family_history', 'stress_immune', 'pulmonary_disease']
   age  gender  smoking  finger_discoloration  mental_stress  \
0   68       1        1                     1              1   
1   81       1        1                     0              0   
2   58       1        1                     0              0   
3   44       0        1                     0              1   
4   72       0        1                     1              1   

   exposure_to_pollution  long_term_illness  energy_level  immune_weakness  \
0                      1                  0     57.831178                0   
1                      1                  1     47.694835                1   
2                      0            

In [3]:
target_col = "pulmonary_disease"

# Binary target: YES -> 1, NO -> 0
y = df[target_col].astype(str).str.upper().map({"NO": 0, "YES": 1})

if y.isna().any():
    invalid_labels = sorted(df.loc[y.isna(), target_col].astype(str).unique().tolist())
    raise ValueError(f"Unexpected target labels found: {invalid_labels}")

X = df.drop(columns=[target_col]).copy()
feature_columns = X.columns.tolist()

print("Features used:", feature_columns)
print("Target balance (YES=1):")
print(y.value_counts(normalize=True))


Features used: ['age', 'gender', 'smoking', 'finger_discoloration', 'mental_stress', 'exposure_to_pollution', 'long_term_illness', 'energy_level', 'immune_weakness', 'breathing_issue', 'alcohol_consumption', 'throat_discomfort', 'oxygen_saturation', 'chest_tightness', 'family_history', 'smoking_family_history', 'stress_immune']
Target balance (YES=1):
pulmonary_disease
0    0.5926
1    0.4074
Name: proportion, dtype: float64


In [4]:
# precaution: keep an unseen test split and use validation for model/threshold selection
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
)


In [5]:
num_imputer = SimpleImputer(strategy="median")

X_train = pd.DataFrame(num_imputer.fit_transform(X_train), columns=X_train.columns)
X_val = pd.DataFrame(num_imputer.transform(X_val), columns=X_val.columns)
X_test = pd.DataFrame(num_imputer.transform(X_test), columns=X_test.columns)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


In [6]:
k_value = min(12, X_train.shape[1])
selector = SelectKBest(score_func=f_classif, k=k_value)

X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_val_selected = selector.transform(X_val_scaled)
X_test_selected = selector.transform(X_test_scaled)

selected_features = [
    col for col, keep in zip(feature_columns, selector.get_support()) if keep
]

print("Selected features:", selected_features)


Selected features: ['smoking', 'finger_discoloration', 'mental_stress', 'exposure_to_pollution', 'energy_level', 'immune_weakness', 'breathing_issue', 'throat_discomfort', 'chest_tightness', 'family_history', 'smoking_family_history', 'stress_immune']


In [7]:
lr_model = LogisticRegression(max_iter=2000, random_state=42, class_weight="balanced")
rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    class_weight="balanced_subsample",
    min_samples_leaf=2,
)

lr_model.fit(X_train_selected, y_train)
rf_model.fit(X_train_selected, y_train)

def evaluate_model(model, X_eval, y_eval):
    y_pred = model.predict(X_eval)
    return {
        "Accuracy": accuracy_score(y_eval, y_pred),
        "Precision": precision_score(y_eval, y_pred, zero_division=0),
        "Recall": recall_score(y_eval, y_pred, zero_division=0),
        "F1": f1_score(y_eval, y_pred, zero_division=0),
    }

lr_results = evaluate_model(lr_model, X_val_selected, y_val)
rf_results = evaluate_model(rf_model, X_val_selected, y_val)

print("\nLogistic Regression (validation):", lr_results)
print("Random Forest (validation):", rf_results)

best_model = rf_model if rf_results["F1"] > lr_results["F1"] else lr_model

print("\nBest Model Selected:",
      "Random Forest" if best_model == rf_model else "Logistic Regression")



Logistic Regression (validation): {'Accuracy': 0.883, 'Precision': 0.8387850467289719, 'Recall': 0.8820638820638821, 'F1': 0.859880239520958}
Random Forest (validation): {'Accuracy': 0.909, 'Precision': 0.9072164948453608, 'Recall': 0.8648648648648649, 'F1': 0.8855345911949686}

Best Model Selected: Random Forest


In [8]:
final_model = best_model

best_threshold = 0.5
best_f1 = 0

y_probs_val = final_model.predict_proba(X_val_selected)[:, 1]

for thresh in np.arange(0.2, 0.901, 0.01):
    y_pred_thresh = (y_probs_val >= thresh).astype(int)
    f1 = f1_score(y_val, y_pred_thresh, zero_division=0)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = float(round(thresh, 2))

print("\nBest Threshold:", best_threshold)
print("Best Validation F1:", best_f1)

y_probs_test = final_model.predict_proba(X_test_selected)[:, 1]
y_final_pred = (y_probs_test >= best_threshold).astype(int)

print("\nFinal Test Evaluation:")
print("Accuracy:", accuracy_score(y_test, y_final_pred))
print("Precision:", precision_score(y_test, y_final_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_final_pred, zero_division=0))
print("F1 Score:", f1_score(y_test, y_final_pred, zero_division=0))



Best Threshold: 0.47
Best Validation F1: 0.8869346733668342

Final Test Evaluation:
Accuracy: 0.899
Precision: 0.8731707317073171
Recall: 0.8796068796068796
F1 Score: 0.8763769889840881


In [9]:
model_data = {
    "model": final_model,
    "imputer": num_imputer,
    "scaler": scaler,
    "selector": selector,
    "threshold": best_threshold,
    "feature_columns": feature_columns,
    "selected_features": selected_features,
    "target_column": target_col,
    "target_definition": "pulmonary_disease == 'YES'",
    "positive_label": "Lung Cancer Risk (YES)",
    "negative_label": "Lung Cancer Risk (NO)",
    "source_dataset": "../data/Lung Cancer.csv",
}

joblib.dump(model_data, "../models/lung_model.pkl")

print("\nModel saved successfully!")



Model saved successfully!
